# 07 - Reference CNN Training Preparation

**Full training is disabled by default.** The first Kaggle Run All performs the unchanged notebook06 setup verification, followed only by a short, disposable GPU training smoke/timing check. Keep both existing inputs attached: `enge707-gpu-setup` (automatically extracted folder or original ZIP) and `enge707-2026-s2`. No installs, weights downloads, AMP, distributed execution or test-image access.

The previously downloaded hosted setup report passed all 40 checks and is preserved unchanged locally as `outputs/metrics/kaggle_gpu_setup_check.json`. Its SHA256 is `e11475982300a0f6aaf4a147968e076e98ec2d9840da91f0ef10e92d4afc36f3`. This notebook re-verifies the current runtime and inputs rather than assuming that previous run remains applicable.

Only after reviewing the hosted smoke report should a later run deliberately enable E0. Seeded runs are not guaranteed bitwise identical across package versions, GPUs or GPU kernels.

In [ ]:
# Safety switch: Run All must not launch the real experiment by default.
RUN_FULL_TRAINING = False

E0_CONFIG = {
    "run_name": "E0_reference_cnn_seed42",
    "seed": 42,
    "max_epochs": 15,
    "early_stopping_patience": 3,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "unweighted CrossEntropyLoss on raw logits",
    "scheduler": {"name": "ReduceLROnPlateau", "mode": "min", "factor": 0.5, "patience": 2, "threshold": 0.0},
    "batch_size": 32,
    "num_workers": 0,
    "drop_last": False,
    "device": "cuda:0",
    "dtype": "float32",
    "amp": False,
    "distributed": False,
}


## 1. Explicit paths, pinned bundle and failure report

Prefer the automatically extracted setup at `/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup/`. Verify its pinned transfer-manifest SHA-256 before trusting its payload checksums. If that directory is absent, retain the original route requiring exactly one matching ZIP under `/kaggle/input` and its full pinned hash. Failures are saved to `/kaggle/working/gpu_setup_check.json`.

In [ ]:
from pathlib import Path, PurePosixPath, PureWindowsPath
from collections import Counter
from importlib.metadata import version
import hashlib
import importlib
import json
import platform
import stat
import struct
import sys
import zipfile

INPUT_ROOT=Path("/kaggle/input")
DATA_ROOT=Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2/")
WORKING_ROOT=Path("/kaggle/working")
SETUP_ROOT=WORKING_ROOT/"enge707_gpu_setup"
REPORT_PATH=WORKING_ROOT/"gpu_setup_check.json"
BUNDLE_NAME="enge707_gpu_setup.zip"
EXTRACTED_BUNDLE_ROOT=Path("/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup/")
EXPECTED_TRANSFER_MANIFEST_SHA256="14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
EXPECTED_BUNDLE_SHA256="e858d40c2aa73096d5eab2fdf8d376f362f33e118924dc7be0269b353c59574f"
EXPECTED_MEMBERS={"src/__init__.py","src/data_pipeline.py","src/models.py",
                  "data/processed/split_manifest.csv","data/processed/class_mapping.json",
                  "preprocessing_fingerprints.json","transfer_manifest.json"}
report={"status":"pending","stage":"initialisation","versions":{"python":platform.python_version()},
        "checks":{},"failures":[],"gpu_check_executed":False}

def save_report():
    WORKING_ROOT.mkdir(parents=True,exist_ok=True)
    REPORT_PATH.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")

def require(name,condition,detail=None):
    report["checks"][name]=bool(condition)
    if not condition:
        raise ValueError(detail or name)
    print(name+": PASS")

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()

save_report()


In [ ]:
try:
    report["stage"]="bundle discovery"
    require("competition_data_root_exists",DATA_ROOT.is_dir())
    if EXTRACTED_BUNDLE_ROOT.exists() or EXTRACTED_BUNDLE_ROOT.is_symlink():
        require("extracted_bundle_is_directory_not_link",EXTRACTED_BUNDLE_ROOT.is_dir() and not EXTRACTED_BUNDLE_ROOT.is_symlink())
        bundle=EXTRACTED_BUNDLE_ROOT
        report["bundle"]={"kind":"extracted_directory","path":str(bundle)}
    else:
        bundles=sorted(path for path in INPUT_ROOT.rglob(BUNDLE_NAME) if path.is_file())
        report["bundle_candidates"]=[str(path) for path in bundles]
        require("exactly_one_attached_original_zip",len(bundles)==1,
                f"No extracted setup directory at {EXTRACTED_BUNDLE_ROOT}; expected one original {BUNDLE_NAME} under {INPUT_ROOT}, found {len(bundles)}: {bundles}.")
        bundle=bundles[0]
        report["bundle"]={"kind":"zip","path":str(bundle),"sha256":file_hash(bundle)}
        require("zip_matches_reviewed_local_sha256",report["bundle"]["sha256"]==EXPECTED_BUNDLE_SHA256)
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 2. Verified setup copy/extraction and every payload checksum

Accept the extracted folder or original ZIP. Only the seven named members are allowed. Reject unexpected files, links, escaping paths, duplicate archive paths and destination conflicts. Verify the pinned manifest before parsing it, then verify its six payload checksums before copying anything. Preflight the entire destination; retain identical existing files and verify destination hashes before importing code. No recursive deletion or image copying is used.

In [ ]:
try:
    def safe_extract_bundle(bundle,destination):
        if destination.is_symlink():
            raise ValueError("Setup directory must not be a symlink")
        destination=destination.resolve()
        if not destination.is_relative_to(WORKING_ROOT.resolve()) or destination==WORKING_ROOT.resolve():
            raise ValueError("Setup directory must stay inside the dedicated working location")
        if bundle.is_symlink() or getattr(bundle,"is_junction",lambda:False)():
            raise ValueError("Bundle root must not be a link")
        if bundle.is_dir():
            bundle_root=bundle.resolve()
            allowed_directories={str(parent) for name in EXPECTED_MEMBERS for parent in PurePosixPath(name).parents if str(parent)!="."}
            source_files={}
            total_bytes=0
            for path in bundle.rglob("*"):
                relative=path.relative_to(bundle).as_posix()
                if path.is_symlink() or getattr(path,"is_junction",lambda:False)():
                    raise ValueError(f"Link in extracted bundle: {relative}")
                if not path.resolve().is_relative_to(bundle_root):
                    raise ValueError(f"Path escapes extracted bundle: {relative}")
                mode=path.lstat().st_mode
                if stat.S_ISDIR(mode):
                    if relative not in allowed_directories:
                        raise ValueError(f"Unexpected bundle directory: {relative}")
                elif stat.S_ISREG(mode):
                    if relative not in EXPECTED_MEMBERS:
                        raise ValueError(f"Unexpected bundle file: {relative}")
                    source_files[relative]=path
                    total_bytes+=path.stat().st_size
                else:
                    raise ValueError(f"Non-regular bundle member: {relative}")
            if set(source_files)!=EXPECTED_MEMBERS or total_bytes>100_000_000:
                raise ValueError("Missing members or unexpectedly large extracted bundle")
            contents={name:path.read_bytes() for name,path in source_files.items()}
        else:
            with zipfile.ZipFile(bundle) as archive:
                infos=archive.infolist()
                names=[item.filename for item in infos]
                if len(names)!=len(set(name.casefold() for name in names)) or set(names)!=EXPECTED_MEMBERS:
                    raise ValueError("Unexpected, missing or duplicate ZIP members")
                if sum(item.file_size for item in infos)>100_000_000:
                    raise ValueError("Unexpectedly large setup archive")
                for item in infos:
                    path=PurePosixPath(item.filename)
                    mode=item.external_attr>>16
                    if (path.is_absolute() or PureWindowsPath(item.filename).drive
                        or ".." in path.parts or "\\" in item.filename or ":" in item.filename
                        or item.is_dir() or stat.S_IFMT(mode) not in (0,stat.S_IFREG)):
                        raise ValueError(f"Unsafe archive member: {item.filename}")
                contents={item.filename:archive.read(item) for item in infos}
        if hashlib.sha256(contents["transfer_manifest.json"]).hexdigest()!=EXPECTED_TRANSFER_MANIFEST_SHA256:
            raise ValueError("Pinned transfer_manifest.json SHA256 mismatch")
        transfer=json.loads(contents["transfer_manifest.json"])
        expected=transfer["sha256"]
        if set(expected)!=EXPECTED_MEMBERS-{"transfer_manifest.json"}:
            raise ValueError("Transfer manifest has an unexpected checksum inventory")
        for name,digest in expected.items():
            if hashlib.sha256(contents[name]).hexdigest()!=digest:
                raise ValueError(f"Transferred checksum mismatch: {name}")
        if destination.exists():
            extras={path.relative_to(destination).as_posix() for path in destination.rglob("*") if path.is_file() or path.is_symlink()}-EXPECTED_MEMBERS
            if extras:
                raise ValueError(f"Unverified files in dedicated setup directory: {sorted(extras)}")
        plan=[]
        for name,data in contents.items():
            target=destination/name
            if not target.resolve().is_relative_to(destination):
                raise ValueError(f"Extraction leaves setup directory: {name}")
            for parent in (target,*target.parents):
                if parent==destination:
                    break
                if parent.is_symlink():
                    raise ValueError(f"Existing symlink in setup path: {parent}")
                if parent!=target and parent.exists() and not parent.is_dir():
                    raise ValueError(f"Non-directory parent: {parent}")
            if target.exists() and (not target.is_file() or target.read_bytes()!=data):
                raise FileExistsError(f"Conflicting existing file: {target}")
            plan.append((target,data))
        for target,data in plan:
            if not target.exists():
                target.parent.mkdir(parents=True,exist_ok=True)
                with target.open("xb") as stream:
                    stream.write(data)
        # The manifest pin covers the manifest itself for both input routes.
        actual={name:file_hash(destination/name) for name in EXPECTED_MEMBERS}
        for name,digest in expected.items():
            if actual[name]!=digest:
                raise ValueError(f"Extracted checksum mismatch: {name}")
        if actual["transfer_manifest.json"]!=hashlib.sha256(contents["transfer_manifest.json"]).hexdigest():
            raise ValueError("Extracted transfer manifest mismatch")
        return transfer,actual

    report["stage"]="safe extraction and artifact verification"
    transfer,artifact_hashes=safe_extract_bundle(bundle,SETUP_ROOT)
    report["artifact_sha256"]=artifact_hashes
    report["bundle"]["transfer_manifest_sha256"]=artifact_hashes["transfer_manifest.json"]
    require("pinned_transfer_manifest_verified",artifact_hashes["transfer_manifest.json"]==EXPECTED_TRANSFER_MANIFEST_SHA256)
    require("all_transferred_checksums_verified_before_import",all(artifact_hashes[name]==digest for name,digest in transfer["sha256"].items()))
    require("expected_bundle_contract",transfer["expected_counts"]=={"total":63660,"train":50927,"validation":12733}
            and transfer["trainable_parameters"]==94280 and transfer["seed"]==42)
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 3. Hosted versions and original code

Use the hosted packages as installed. Do not install the local requirements or upgrade packages. Report actual versions and their differences from the local CPU runtime. Source checksums have already passed. Disable new bytecode files in the dedicated setup directory, and reject a conflicting preloaded `src` package.

In [ ]:
try:
    report["stage"]="hosted runtime and verified imports"
    for distribution in ["torch","torchvision","numpy","pandas","Pillow"]:
        report["versions"][distribution]=version(distribution)
    report["source_versions"]=transfer["source_versions"]
    report["version_differences"]={name:{"source":value,"hosted":report["versions"].get(name)}
                                   for name,value in transfer["source_versions"].items() if report["versions"].get(name)!=value}
    save_report()
    # Avoid importing stale code from a different input or previous setup directory.
    for name,relative in [("src.models","src/models.py"),("src.data_pipeline","src/data_pipeline.py"),("src","src/__init__.py")]:
        loaded=sys.modules.get(name)
        if loaded is not None:
            if Path(getattr(loaded,"__file__","")).resolve()!=(SETUP_ROOT/relative).resolve():
                raise RuntimeError("A different src package is already loaded. Restart the session.")
            del sys.modules[name]
    sys.dont_write_bytecode=True
    sys.path.insert(0,str(SETUP_ROOT))
    importlib.invalidate_caches()
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    from PIL import Image
    from src.data_pipeline import FacialFeatureDataset, reference_preprocess, seed_everything
    from src.models import ReferenceCNN
    report["cuda_build"]=torch.version.cuda
    print(json.dumps(report["versions"],indent=2))
    print("Version differences:",json.dumps(report["version_differences"],indent=2))
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 4. Discover training nesting and verify the saved split

Locate `train_labels.csv` uniquely. Preserve all original string IDs and exact classes. Find candidate image directories using the largest saved ID as an anchor, then require exactly one directory with the complete expected filename set. This discovers nesting without assuming `train/train`. Only directory entries are examined during discovery; no test image contents are opened. All image decoding later is restricted to verified training IDs.

In [ ]:
try:
    report["stage"]="labels, training directory and fixed split"
    split=pd.read_csv(SETUP_ROOT/"data/processed/split_manifest.csv",dtype=str,keep_default_na=False,na_filter=False)
    mapping=json.loads((SETUP_ROOT/"data/processed/class_mapping.json").read_text(encoding="utf-8"))
    label_paths=sorted(path for path in DATA_ROOT.rglob("train_labels.csv") if path.is_file())
    require("unique_training_labels_csv",len(label_paths)==1)
    labels=pd.read_csv(label_paths[0],dtype=str,keep_default_na=False,na_filter=False)
    require("exact_training_label_columns",list(labels.columns)==["image_id","class"])
    require("original_ids_preserved_unique",labels["image_id"].is_unique and split["image_id"].is_unique
            and labels["image_id"].str.fullmatch(r"[0-9]{5}").all() and split["image_id"].str.fullmatch(r"[0-9]{5}").all())
    require("every_original_id_class_matches",labels.set_index("image_id").sort_index().equals(split[["image_id","class"]].set_index("image_id").sort_index()))
    actual_counts={"total":len(split),**{name:int(split["split"].eq(name).sum()) for name in ["train","validation"]}}
    require("saved_split_counts_and_full_coverage",actual_counts==transfer["expected_counts"] and split["split"].isin(["train","validation"]).all())
    classes=sorted(labels["class"].unique().tolist())
    expected_mapping={"class_to_index":{label:index for index,label in enumerate(classes)},
                      "index_to_class":{str(index):label for index,label in enumerate(classes)}}
    require("saved_class_encoding_unchanged",len(classes)==8 and mapping==transfer["classes"]==expected_mapping)
    require("all_classes_in_both_partitions",all(set(split.loc[split["split"].eq(name),"class"])==set(classes) for name in ["train","validation"]))
    require("valid_saved_hashes",all(split[column].str.fullmatch(r"[0-9a-f]{64}").all() for column in ["byte_sha256","pixel_sha256"]))
    require("original_manifest_paths_retained",split["image_path"].equals("data/raw/train/train/"+split["image_id"]+".jpg"))
    require("saved_pixel_groups_and_labels",split["group_id"].equals(split["pixel_sha256"])
            and split["pixel_sha256"].nunique()==transfer["pixel_groups"]
            and split.groupby("pixel_sha256")["class"].nunique().eq(1).all()
            and split.groupby("pixel_sha256")["split"].nunique().eq(1).all())
    train_rows=split[split["split"].eq("train")]
    validation_rows=split[split["split"].eq("validation")]
    overlaps={column:len(set(train_rows[column])&set(validation_rows[column])) for column in ["image_id","byte_sha256","pixel_sha256"]}
    report["cross_partition_overlap_counts"]=overlaps
    require("zero_cross_partition_overlap",not any(overlaps.values()))
    expected_filenames={image_id+".jpg" for image_id in labels["image_id"]}
    anchor=max(labels["image_id"])+".jpg"
    candidates=sorted({path.parent.resolve() for path in DATA_ROOT.rglob(anchor) if path.is_file()})
    matching=[]
    for directory in candidates:
        filenames={path.name for path in directory.glob("*.jpg") if path.is_file()}
        if filenames==expected_filenames:
            matching.append(directory)
    report["training_directory_candidates"]=[str(path) for path in candidates]
    require("exactly_one_complete_training_image_directory",len(matching)==1)
    TRAIN_IMAGE_DIR=matching[0]
    require("every_training_path_exists",all((TRAIN_IMAGE_DIR/name).is_file() and (TRAIN_IMAGE_DIR/name).resolve().parent==TRAIN_IMAGE_DIR for name in expected_filenames))
    train_dataset=FacialFeatureDataset(SETUP_ROOT,"train",training_image_directory=TRAIN_IMAGE_DIR)
    validation_dataset=FacialFeatureDataset(SETUP_ROOT,"validation",training_image_directory=TRAIN_IMAGE_DIR)
    require("portable_dataset_lengths_match",len(train_dataset)==50927 and len(validation_dataset)==12733)
    report["training_directory"]=str(TRAIN_IMAGE_DIR)
    report["training_labels_path"]=str(label_paths[0])
    report["counts"]=actual_counts
    report["mapping"]=mapping
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 5. Strict fingerprints before enabling the GPU check

Recompute byte and reference-preprocessed **CPU float32** tensor hashes on the same 12 notebook03 training images. Use the recorded shape, dtype and canonical hashing method. Report all differences and actual runtime versions. A mismatch stops the GPU check; do not change transforms, labels or tolerances to hide it.

In [ ]:
try:
    report["stage"]="strict preprocessing fingerprints"
    fingerprint_spec=json.loads((SETUP_ROOT/"preprocessing_fingerprints.json").read_text(encoding="utf-8"))
    examples=fingerprint_spec["examples"]
    require("twelve_unique_training_fingerprints",len(examples)==12 and len({item["image_id"] for item in examples})==12)
    fingerprint_results=[]
    for expected in examples:
        row=train_rows.loc[train_rows["image_id"].eq(expected["image_id"])]
        require("fingerprint_training_membership_"+expected["image_id"],len(row)==1 and row.iloc[0]["class"]==expected["class"])
        image_path=TRAIN_IMAGE_DIR/(expected["image_id"]+".jpg")
        actual_byte_hash=file_hash(image_path)
        with Image.open(image_path) as image:
            image.load()
            tensor=reference_preprocess(image)
        header=b"ENGE707-reference-f32-v1\0"+struct.pack(">3Q",*tensor.shape)
        pixels=np.asarray(tensor.detach().cpu().contiguous().numpy(),dtype="<f4")
        actual_tensor_hash=hashlib.sha256(header+pixels.tobytes(order="C")).hexdigest()
        fingerprint_results.append({"image_id":expected["image_id"],"original_byte_sha256":actual_byte_hash,
            "expected_original_byte_sha256":expected["original_byte_sha256"],"tensor_sha256":actual_tensor_hash,
            "expected_tensor_sha256":expected["tensor_sha256"],"shape":list(tensor.shape),"dtype":str(tensor.dtype),
            "byte_match":actual_byte_hash==expected["original_byte_sha256"],
            "tensor_match":actual_tensor_hash==expected["tensor_sha256"],
            "contract_match":list(tensor.shape)==expected["shape"]==[3,224,224] and tensor.dtype==torch.float32 and expected["dtype"]=="float32" and tensor.device.type=="cpu"})
    report["fingerprints"]=fingerprint_results
    report["fingerprint_method"]=fingerprint_spec["method"]
    save_report()
    require("all_strict_fingerprints_match",all(item["byte_match"] and item["tensor_match"] and item["contract_match"] for item in fingerprint_results))
    report["gpu_check_enabled"]=True
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 6. Fresh original model: one inference-only batch on cuda:0

Select cuda:0 only, even if multiple GPUs are visible. Instantiate the original model fresh with seed 42. Take four verified training-partition images and use reference preprocessing directly, with no augmentation. Run one evaluation forward pass under `torch.inference_mode()`; do not create an optimiser or run backward. No weights are saved.

In [ ]:
try:
    report["stage"]="cuda:0 inference verification"
    require("fingerprints_gate_gpu_check",report.get("gpu_check_enabled",False))
    report["cuda_available"]=torch.cuda.is_available()
    report["cuda_device_count"]=torch.cuda.device_count()
    save_report()
    require("cuda_available",report["cuda_available"] and report["cuda_device_count"]>=1)
    torch.cuda.set_device(0)
    device=torch.device("cuda:0")
    free_bytes,total_bytes=torch.cuda.mem_get_info(0)
    report["gpu"]={"selected_device":"cuda:0","name":torch.cuda.get_device_name(0),
                   "available_memory_bytes":free_bytes,"total_memory_bytes":total_bytes}
    seed_everything(42)
    model=ReferenceCNN().to(device)
    parameter_count=sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    report["trainable_parameters"]=parameter_count
    require("original_parameter_count",parameter_count==94280==transfer["trainable_parameters"])
    require("model_uses_cuda0_only",all(parameter.device==device for parameter in model.parameters()))
    sample_rows=train_rows.head(4)
    tensors=[]
    for image_id in sample_rows["image_id"]:
        with Image.open(TRAIN_IMAGE_DIR/(image_id+".jpg")) as image:
            image.load()
            tensors.append(reference_preprocess(image))
    batch=torch.stack(tensors).to(device)
    model.eval()
    with torch.inference_mode():
        logits=model(batch)
    report["gpu_check_executed"]=True
    report["inference"]={"image_ids":sample_rows["image_id"].tolist(),"batch_shape":list(batch.shape),
                         "logit_shape":list(logits.shape),"device":str(logits.device),"finite":bool(torch.isfinite(logits).all()),
                         "requires_grad":logits.requires_grad}
    require("finite_eight_logit_output",list(logits.shape)==[len(sample_rows),8] and report["inference"]["finite"])
    require("eval_and_inference_only",not model.training and not logits.requires_grad and logits.is_inference())
    require("transferred_artifacts_remain_unchanged",all(file_hash(SETUP_ROOT/name)==digest for name,digest in artifact_hashes.items()))
    report["status"]="passed"
    save_report()
    del model,batch,logits,tensors
    print("Saved actual hosted results:",REPORT_PATH)
    print("GPU preparation checks passed. No training was performed.")
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 7. Sample-weighted training and evaluation helpers

Cross-entropy receives the raw logits with no softmax or class weights. Accumulate `batch_mean_loss * actual_batch_size`, correct predictions and sample totals; divide only at the end. This correctly includes the final partial batch. Loading errors propagate. Non-finite logits, loss or gradients raise an error instead of being skipped. Validation uses both evaluation mode and inference mode.

In [ ]:
import time
import statistics
from torch import nn
from torch.utils.data import DataLoader

def train_batch(model,batch,optimizer,criterion,device,check_update=False):
    """One float32 step; returns sample totals, not an experimental result."""
    model.train()
    images,targets,_=batch
    images=images.to(device=device,dtype=torch.float32)
    targets=targets.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits=model(images)
    loss=criterion(logits,targets)
    if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(loss)):
        raise FloatingPointError("Non-finite training logits or cross-entropy")
    loss.backward()
    gradients=[parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    if not gradients or not all(bool(torch.isfinite(gradient).all()) for gradient in gradients):
        raise FloatingPointError("Missing or non-finite training gradients")
    nonzero=any(bool(torch.any(gradient!=0)) for gradient in gradients)
    old_parameters=[parameter.detach().clone() for parameter in model.parameters()] if check_update else None
    optimizer.step()
    changed=None
    if check_update:
        changed=any(not torch.equal(before,after.detach()) for before,after in zip(old_parameters,model.parameters()))
        if not changed:
            raise RuntimeError("Smoke optimiser step did not change any parameter")
    size=int(targets.shape[0])
    return {"loss_sum":float(loss.detach())*size,"correct":int((logits.detach().argmax(dim=1)==targets).sum()),
            "samples":size,"finite_gradients":True,"nonzero_gradient":nonzero,"parameters_changed":changed}


def evaluate_batch(model,batch,criterion,device):
    model.eval()
    with torch.inference_mode():
        images,targets,_=batch
        images=images.to(device=device,dtype=torch.float32)
        targets=targets.to(device)
        logits=model(images)
        loss=criterion(logits,targets)
        if not bool(torch.isfinite(logits).all()) or not bool(torch.isfinite(loss)):
            raise FloatingPointError("Non-finite validation logits or cross-entropy")
        size=int(targets.shape[0])
        return {"loss_sum":float(loss)*size,"correct":int((logits.argmax(dim=1)==targets).sum()),"samples":size}


def train_epoch(model,loader,optimizer,criterion,device):
    total_loss=0.0
    total_correct=total_samples=0
    for batch in loader:
        result=train_batch(model,batch,optimizer,criterion,device)
        total_loss+=result["loss_sum"]
        total_correct+=result["correct"]
        total_samples+=result["samples"]
    if total_samples!=len(loader.dataset) or total_samples==0:
        raise ValueError("Training did not cover all saved training rows exactly once")
    return {"loss":total_loss/total_samples,"accuracy":total_correct/total_samples,"samples":total_samples}


def evaluate_epoch(model,loader,criterion,device):
    total_loss=0.0
    total_correct=total_samples=0
    for batch in loader:
        result=evaluate_batch(model,batch,criterion,device)
        total_loss+=result["loss_sum"]
        total_correct+=result["correct"]
        total_samples+=result["samples"]
    if total_samples!=len(loader.dataset) or total_samples==0:
        raise ValueError("Validation did not cover all saved validation rows")
    return {"loss":total_loss/total_samples,"accuracy":total_correct/total_samples,"samples":total_samples}


In [ ]:
# Prepare a verified raw-JPEG working copy before the disposable GPU smoke test.
# This tests a storage hypothesis; no decoded images or transforms are cached.
import hashlib
import json
import re
import shutil
import time
from pathlib import Path


def prepare_training_image_cache(manifest, source_directory, cache_directory,
                                 summary_path, expected_count=63660):
    source = Path(source_directory).absolute()
    cache = Path(cache_directory).absolute()
    summary_path = Path(summary_path).absolute()
    # Reject root links before resolution; individual files are checked below.
    if source.is_symlink() or cache.is_symlink() or summary_path.is_symlink():
        raise ValueError("Source/cache/report must not be symbolic links")
    if not source.is_dir() or not cache.parent.is_dir():
        raise ValueError("Source and cache parent directories must exist")
    source = source.resolve()
    if cache.resolve().parent != cache.parent.resolve():
        raise ValueError("Cache root escapes its intended parent")
    cache = cache.resolve()
    if source == cache or source in cache.parents or cache in source.parents:
        raise ValueError("Source and cache must be separate folders")
    if summary_path.parent.resolve() != cache.parent or summary_path == cache:
        raise ValueError("Cache report must stay beside the cache folder")
    started = time.perf_counter()
    summary = {
        "status": "running", "source_directory": str(source),
        "cache_directory": str(cache), "expected_files": expected_count,
        "copied_files": 0, "reused_files": 0, "verified_files": 0,
        "hash_checks": 0, "hash_mismatches": 0, "failures": [],
        "hash_method": "SHA256 of original JPEG file bytes, matched to byte_sha256",
        "checks": {}, "disk_space_before": None, "disk_space_after": None,
    }

    def checked_path(root, filename):
        path = root / filename
        if path.is_symlink() or path.resolve().parent != root:
            raise ValueError(f"Unsafe image path: {path}")
        if path.exists() and not path.is_file():
            raise ValueError(f"Image path is not a regular file: {path}")
        return path

    def verify(path, expected):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        summary["hash_checks"] += 1
        if digest.hexdigest() != expected:
            summary["hash_mismatches"] += 1
            raise ValueError(f"SHA256 conflict (left unchanged): {path}")
        summary["verified_files"] += 1

    try:
        if len(manifest) != expected_count or not manifest["image_id"].is_unique:
            raise ValueError("Expected complete manifest with unique original IDs")
        entries = list(manifest[["image_id", "byte_sha256"]].itertuples(index=False, name=None))
        if not all(isinstance(i, str) and re.fullmatch(r"[0-9]{5}", i)
                   and isinstance(h, str) and re.fullmatch(r"[0-9a-f]{64}", h)
                   for i, h in entries):
            raise ValueError("Invalid original string IDs or manifest byte hashes")
        filenames = {image_id + ".jpg" for image_id, _ in entries}
        if cache.exists():
            if not cache.is_dir():
                raise ValueError("Cache path is not a directory")
            for path in cache.iterdir():
                if path.name not in filenames:
                    raise ValueError(f"Unexpected cache entry (left unchanged): {path}")
                checked_path(cache, path.name)
        summary["checks"]["manifest_ids_and_cache_paths_valid"] = True
        disk = shutil.disk_usage(cache.parent)
        summary["disk_space_before"] = dict(zip(["total", "used", "free"], disk))

        # Preflight every path and existing-file hash before creating any files.
        pending = []
        source_bytes = pending_bytes = 0
        for number, (image_id, expected_hash) in enumerate(entries, 1):
            filename = image_id + ".jpg"
            original = checked_path(source, filename)
            if not original.is_file():
                raise FileNotFoundError(original)
            destination = checked_path(cache, filename)
            size = original.stat().st_size
            source_bytes += size
            if destination.exists():
                verify(destination, expected_hash)
                summary["reused_files"] += 1
            else:
                pending.append((filename, expected_hash))
                pending_bytes += size
            if number % 10000 == 0:
                print(f"Cache preflight: {number:,}/{expected_count:,} files", flush=True)
        # Allow for per-file allocation overhead and a modest free-space reserve.
        overhead = 4096 * len(pending)
        reserve = 256 * 1024 * 1024 if pending else 0
        required = pending_bytes + overhead + reserve
        summary.update(source_bytes=source_bytes, bytes_to_copy=pending_bytes,
                       files_to_copy=len(pending), estimated_file_overhead_bytes=overhead,
                       free_space_reserve_bytes=reserve, required_free_bytes=required)
        if disk.free < required:
            raise OSError(f"Insufficient disk: need {required:,} bytes; free {disk.free:,}")
        summary["checks"]["free_space_sufficient"] = True
        cache.mkdir(exist_ok=True)
        for number, (filename, expected_hash) in enumerate(pending, 1):
            original = checked_path(source, filename)
            destination = checked_path(cache, filename)
            # Exclusive creation: never overwrite a conflict or interrupted copy.
            with original.open("rb") as incoming, destination.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing, length=1024 * 1024)
            summary["copied_files"] += 1
            verify(destination, expected_hash)
            if number % 5000 == 0 or number == len(pending):
                print(f"Copied and SHA256 verified: {number:,}/{len(pending):,} new files", flush=True)
        final_names = {path.name for path in cache.iterdir()}
        if final_names != filenames or summary["verified_files"] != expected_count:
            raise ValueError("Incomplete cache coverage")
        summary["checks"].update(all_expected_files_present=True, every_file_hash_matches=True)
        summary["status"] = "passed"
    except Exception as error:
        summary["status"] = "failed"
        summary["failures"].append(f"{type(error).__name__}: {error}")
        raise
    finally:
        summary["preparation_seconds"] = time.perf_counter() - started
        summary["disk_space_after"] = dict(zip(
            ["total", "used", "free"], shutil.disk_usage(cache.parent)))
        summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
        print("Cache verification:", summary["status"], "--", summary_path, flush=True)
    return summary


training_cache_check = {}  # A failed/unfinished preparation must block later smoke/E0.
if report.get("status") != "passed" or not report.get("gpu_check_executed"):
    raise RuntimeError("Complete the existing verified setup before cache preparation")
# The setup report retains the original input path even after TRAIN_IMAGE_DIR changes.
if "ORIGINAL_TRAIN_IMAGE_DIR" not in globals():
    ORIGINAL_TRAIN_IMAGE_DIR = Path(report["training_directory"]).resolve()
if ORIGINAL_TRAIN_IMAGE_DIR != Path(report["training_directory"]).resolve():
    raise ValueError("Original verified training directory changed")
TRAIN_CACHE_DIR = WORKING_ROOT / "enge707_train_cache"
TRAIN_CACHE_REPORT = WORKING_ROOT / "training_image_cache_check.json"
if not all(file_hash(SETUP_ROOT / name) == artifact_hashes[name] for name in [
        "data/processed/split_manifest.csv", "data/processed/class_mapping.json"]):
    raise ValueError("Saved split or class mapping changed")
# Read the saved manifest again as strings; do not regenerate membership or hashes.
cache_manifest = pd.read_csv(SETUP_ROOT / "data/processed/split_manifest.csv",
                            dtype=str, keep_default_na=False, na_filter=False)
cache_verification = prepare_training_image_cache(
    cache_manifest, ORIGINAL_TRAIN_IMAGE_DIR, TRAIN_CACHE_DIR, TRAIN_CACHE_REPORT)
cached_train_dataset = FacialFeatureDataset(SETUP_ROOT, "train", training_image_directory=TRAIN_CACHE_DIR)
cached_validation_dataset = FacialFeatureDataset(SETUP_ROOT, "validation", training_image_directory=TRAIN_CACHE_DIR)
if (len(cached_train_dataset) != 50927 or len(cached_validation_dataset) != 12733
        or not cached_train_dataset.rows.equals(train_dataset.rows)
        or not cached_validation_dataset.rows.equals(validation_dataset.rows)):
    raise ValueError("Working-copy dataset membership/labels/order changed")
# Switch both partitions only after all files and unchanged dataset metadata pass.
TRAIN_IMAGE_DIR = TRAIN_CACHE_DIR.resolve()
train_dataset, validation_dataset = cached_train_dataset, cached_validation_dataset
training_cache_check = cache_verification
print("Both saved partitions now use verified raw JPEGs:", TRAIN_IMAGE_DIR)

## 8. Disposable GPU smoke/timing phase (always bounded)

Reset seeds to 42 and create a disposable original model, Adam optimiser and fresh real loaders. Request two warm-up training batches, then up to five timed training batches and three validation batches. Synchronise cuda:0 before and after each measured batch and include loading inside its measured interval. A roughly 120-second budget includes warm-up; check it between batches and reserve 30 seconds for validation. One unexpectedly slow batch can overrun the budget.

Check finite gradients and at least one changed parameter. Save actual completions, individual times, runtime/GPU details and a rough epoch projection to `reference_gpu_smoke_check.json`. Smoke diagnostics are not experimental performance results. Discard the model and optimiser even on failure. No smoke checkpoint, history or learning curves are saved.

In [ ]:
SMOKE_PATH=WORKING_ROOT/"reference_gpu_smoke_check.json"
smoke={"status":"running","diagnostic_only":True,"seed":42,"device":"cuda:0","dtype":"float32",
       "versions":dict(report["versions"]),"cuda_build":torch.version.cuda,"gpu":dict(report["gpu"]),
       "configuration":{"batch_size":32,"num_workers":0,"drop_last":False,"train_shuffle":True,
                        "validation_shuffle":False,"optimizer":"Adam","learning_rate":0.001,
                        "loss":"unweighted CrossEntropyLoss","amp":False,"distributed":False},
       "budget_seconds":120.0,"validation_reserve_seconds":30.0,
       "batch_seconds":{"warmup":[],"training":[],"validation":[]},
       "completed_batches":{"warmup":0,"training":0,"validation":0},
       "completed_samples":{"warmup":0,"training":0,"validation":0},
       "checks":{},"failures":[],"stop_reasons":[]}

def save_smoke():
    SMOKE_PATH.write_text(json.dumps(smoke,indent=2)+"\n",encoding="utf-8")

smoke_model=smoke_optimizer=None
smoke_started=None
save_smoke()
try:
    if report["status"]!="passed" or not report["gpu_check_executed"] or device!=torch.device("cuda:0"):
        raise RuntimeError("Complete the unchanged setup and cuda:0 checks before smoke testing")
    if (globals().get("training_cache_check", {}).get("status") != "passed"
            or TRAIN_IMAGE_DIR != TRAIN_CACHE_DIR.resolve()
            or any(dataset.training_directory != TRAIN_IMAGE_DIR
                   for dataset in [train_dataset, validation_dataset])):
        raise RuntimeError("Complete verified working-copy preparation before smoke testing")
    smoke["image_storage"] = {"original_directory": str(ORIGINAL_TRAIN_IMAGE_DIR),
        "active_directory": str(TRAIN_IMAGE_DIR), "cache_report": str(TRAIN_CACHE_REPORT),
        "verified_files": training_cache_check["verified_files"]}
    seed_everything(42)
    smoke_model=ReferenceCNN().to(device=device,dtype=torch.float32)
    if sum(p.numel() for p in smoke_model.parameters() if p.requires_grad)!=94280:
        raise ValueError("Reference architecture changed")
    smoke_optimizer=torch.optim.Adam(smoke_model.parameters(),lr=0.001)
    smoke_criterion=nn.CrossEntropyLoss()
    smoke_train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,
                                 generator=torch.Generator().manual_seed(42),num_workers=0,drop_last=False)
    smoke_validation_loader=DataLoader(validation_dataset,batch_size=32,shuffle=False,num_workers=0,drop_last=False)
    smoke["loader_batch_counts"]={"training":len(smoke_train_loader),"validation":len(smoke_validation_loader)}
    free_bytes,total_bytes=torch.cuda.mem_get_info(0)
    smoke["gpu"].update(available_memory_before_bytes=free_bytes,total_memory_bytes=total_bytes)
    torch.cuda.synchronize(device)
    smoke_started=time.perf_counter()
    smoke_training_iterator=iter(smoke_train_loader)
    changed_step=False
    nonzero_seen=False
    finite_gradients_seen=False
    for phase,maximum in [("warmup",2),("training",5),("validation",3)]:
        if phase=="training" and smoke["completed_batches"]["warmup"]<2:
            smoke["stop_reasons"].append("Timed training omitted: two warm-ups did not fit")
            continue
        if phase=="validation":
            if time.perf_counter()-smoke_started>=120:
                smoke["stop_reasons"].append("Validation omitted: budget exhausted")
                break
            smoke_validation_iterator=iter(smoke_validation_loader)
        for batch_number in range(maximum):
            elapsed=time.perf_counter()-smoke_started
            previous_times=smoke["batch_seconds"][phase]
            estimate=previous_times[-1] if previous_times else (
                smoke["batch_seconds"]["warmup"][-1] if phase=="training" and smoke["batch_seconds"]["warmup"] else 0.0)
            reserve=30.0 if phase=="training" else 0.0
            if elapsed+estimate>=120.0-reserve:
                smoke["stop_reasons"].append(f"Stopped {phase} before next batch: time budget/reserve")
                break
            torch.cuda.synchronize(device)
            batch_start=time.perf_counter()
            if phase=="validation":
                result=evaluate_batch(smoke_model,next(smoke_validation_iterator),smoke_criterion,device)
            else:
                result=train_batch(smoke_model,next(smoke_training_iterator),smoke_optimizer,smoke_criterion,
                                   device,check_update=not changed_step)
                changed_step=changed_step or result["parameters_changed"] is True
                nonzero_seen=nonzero_seen or result["nonzero_gradient"]
                finite_gradients_seen=finite_gradients_seen or result["finite_gradients"]
            torch.cuda.synchronize(device)
            duration=time.perf_counter()-batch_start
            smoke["batch_seconds"][phase].append(duration)
            smoke["completed_batches"][phase]+=1
            smoke["completed_samples"][phase]+=result["samples"]
            print(f"{phase} batch {batch_number+1}: {duration:.3f}s (loading included)",flush=True)
            save_smoke()
    smoke["timing_elapsed_seconds"]=time.perf_counter()-smoke_started
    smoke["budget_overrun_seconds"]=max(0.0,smoke["timing_elapsed_seconds"]-120.0)
    means={phase:statistics.mean(times) if times else None for phase,times in smoke["batch_seconds"].items()}
    smoke["mean_batch_seconds"]=means
    estimate=(means["training"]*len(smoke_train_loader)+means["validation"]*len(smoke_validation_loader)
              if means["training"] is not None and means["validation"] is not None else None)
    smoke["rough_epoch_projection"]={"label":"Rough short-sample projection, not a measured epoch",
        "formula":"mean training batch * actual training batches + mean validation batch * actual validation batches",
        "seconds":estimate,"minutes":estimate/60 if estimate is not None else None}
    smoke["checks"]={"finite_losses_and_gradients":finite_gradients_seen,"some_gradient_nonzero":nonzero_seen,
                     "optimizer_changed_weights":changed_step,"two_warmup_batches":smoke["completed_batches"]["warmup"]==2,
                     "at_least_one_timed_batch_each":smoke["completed_batches"]["training"]>=1 and smoke["completed_batches"]["validation"]>=1,
                     "bounded_counts":smoke["completed_batches"]["training"]<=5 and smoke["completed_batches"]["validation"]<=3,
                     "transferred_artifacts_unchanged":all(file_hash(SETUP_ROOT/name)==digest for name,digest in artifact_hashes.items())}
    smoke["failed_checks"]=[name for name,passed in smoke["checks"].items() if not passed]
    smoke["status"]="passed" if not smoke["failed_checks"] else "incomplete"
    smoke["limitations"]=["Smoke values are diagnostics, not experimental performance results.",
        "Working-copy loading, finite checks and warm cache state affect this short-sample projection; sustained GPU speed may differ.",
        "Partial final batches are projected using full-batch means; warm-up is excluded from the projected epoch.",
        "No cross-version or GPU bitwise reproducibility is promised."]
except Exception as error:
    smoke["status"]="failed"
    smoke["failures"].append(f"{type(error).__name__}: {error}")
    if smoke_started is not None:
        smoke["timing_elapsed_seconds"]=time.perf_counter()-smoke_started
    raise
finally:
    smoke_model=smoke_optimizer=None
    smoke["disposable_model_and_optimizer_discarded"]=True
    save_smoke()
print("Saved GPU diagnostic report:",SMOKE_PATH)
print("Smoke status:",smoke["status"],"-- not an experimental result")


In [ ]:
# Optional bounded pipeline diagnostic: paste/run after the successful GPU smoke cell.
# Full training stays disabled. All model updates below are disposable diagnostics.
import json
import time
import statistics
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset


def _run_gpu_pipeline_timing_check():
    if RUN_FULL_TRAINING is not False:
        raise RuntimeError("Set RUN_FULL_TRAINING = False before this diagnostic")
    if report.get("status") != "passed" or smoke.get("status") != "passed":
        raise RuntimeError("Complete the existing setup and GPU smoke checks first")
    if not torch.cuda.is_available():
        raise RuntimeError("This diagnostic requires the verified Kaggle cuda:0 runtime")

    output = Path("/kaggle/working/gpu_pipeline_timing_check.json")
    gpu = torch.device("cuda:0")
    original_threads = torch.get_num_threads()
    started = time.perf_counter()
    result = {
        "status": "running", "diagnostic_only": True, "budget_seconds": 90,
        "device": str(gpu), "gpu_name": torch.cuda.get_device_name(0),
        "torch_version": str(torch.__version__), "cuda_build": torch.version.cuda,
        "runtime_versions": dict(report.get("versions", {})),
        "cpu_threads": original_threads,
        "cpu_interop_threads": torch.get_num_interop_threads(),
        "configuration": {
            "seed": 42, "batch_size": 32, "num_workers": 0, "drop_last": False,
            "pin_memory": False, "non_blocking_transfer": False,
            "train_shuffle": True, "validation_shuffle": False, "dtype": "float32",
            "optimizer": "Adam", "learning_rate": 0.001,
            "loss": "unweighted CrossEntropyLoss", "amp": False,
            "training_augmentation": repr(train_dataset.augmentation),
            "preprocessing": "unchanged reference RGB resize/pad then float32 /255",
            "dataset_lengths": [len(train_dataset), len(validation_dataset)],
        },
        "pipeline_batches": [], "resident_gpu_step_seconds": [],
        "resident_warmup_steps": 0, "thread_loading_samples": [],
        "stop_reasons": [], "failures": [],
        "limitations": [
            "Short diagnostics only; no measured epoch or experimental accuracy.",
            "Fetch includes decode, transforms and collation; num_workers=0.",
            "Synchronized stages are serialized; compute includes finite checks.",
            "Resident steps reuse one GPU batch and exclude loading and transfer.",
            "Thread comparison uses two matched 32-image training samples, with "
            "opposite setting orders; augmentation seeds match within each pair.",
            "Filesystem/cache state and fixed sample/order effects are uncontrolled; "
            "this comparison cannot establish the cause of the earlier timings.",
            "The budget is checked between batches; one operation can overrun it.",
        ],
    }
    model = optimizer = resident = None

    def room(deadline, estimate=0.0):
        if time.perf_counter() - started + estimate >= deadline:
            result["stop_reasons"].append(f"Reached {deadline}s phase budget")
            return False
        return True

    def timed(operation):
        torch.cuda.synchronize(gpu)
        start = time.perf_counter()
        value = operation()
        torch.cuda.synchronize(gpu)
        return value, time.perf_counter() - start

    def compute(images, labels, training):
        model.train(training)
        if training:
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not bool(torch.isfinite(logits).all() & torch.isfinite(loss)):
                raise FloatingPointError("Nonfinite training logits/loss")
            loss.backward()
            gradients = [p.grad for p in model.parameters() if p.grad is not None]
            if not gradients or not all(bool(torch.isfinite(g).all()) for g in gradients):
                raise FloatingPointError("Missing/nonfinite gradients")
            optimizer.step()
        else:
            with torch.inference_mode():
                logits = model(images)
                loss = criterion(logits, labels)
                if not bool(torch.isfinite(logits).all() & torch.isfinite(loss)):
                    raise FloatingPointError("Nonfinite validation logits/loss")

    try:
        seed_everything(42)
        model = ReferenceCNN().to(device=gpu, dtype=torch.float32)
        result["trainable_parameters"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if result["trainable_parameters"] != 94280:
            raise ValueError("Reference architecture changed")
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
            generator=torch.Generator().manual_seed(42), num_workers=0, drop_last=False)
        validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False,
            num_workers=0, drop_last=False)
        train_iterator, validation_iterator = iter(train_loader), iter(validation_loader)

        # One excluded warm-up, then at most two batches per partition.
        previous = 0.0
        for phase in ["warmup", "training", "training", "validation", "validation"]:
            if not room(50, previous):
                break
            iterator = validation_iterator if phase == "validation" else train_iterator
            batch, fetch_seconds = timed(lambda: next(iterator))
            images, labels, ids = batch
            if images.shape != (32, 3, 224, 224) or images.dtype != torch.float32:
                raise ValueError("Unexpected input shape/precision")
            if labels.dtype != torch.int64:
                raise ValueError("Unexpected label dtype")
            moved, transfer_seconds = timed(lambda: (
                images.to(device=gpu, dtype=torch.float32), labels.to(device=gpu)))
            _, compute_seconds = timed(lambda: compute(*moved, phase != "validation"))
            if phase != "validation":
                resident = moved
            previous = fetch_seconds + transfer_seconds + compute_seconds
            result["pipeline_batches"].append({
                "phase": phase, "samples": len(ids), "image_ids": list(ids),
                "fetch_seconds": fetch_seconds, "transfer_seconds": transfer_seconds,
                "compute_seconds": compute_seconds, "stage_sum_seconds": previous,
            })

        # Same already-resident training batch; warm up before three timed steps.
        if resident is not None:
            previous = 0.0
            for step in range(5):
                if not room(65, previous):
                    break
                _, previous = timed(lambda: compute(*resident, True))
                if step < 2:
                    result["resident_warmup_steps"] += 1
                else:
                    result["resident_gpu_step_seconds"].append(previous)

        # Matched small samples: current->1, then 1->current to vary cache order.
        if original_threads > 1:
            previous = 0.0
            for sample, settings in enumerate([(original_threads, 1), (1, original_threads)]):
                for threads in settings:
                    if not room(90, previous):
                        break
                    torch.set_num_threads(threads)
                    if not room(90, previous):
                        break
                    seed_everything(42 + sample)
                    indices = list(range(sample * 32, (sample + 1) * 32))
                    loader = DataLoader(Subset(train_dataset, indices), batch_size=32,
                        shuffle=False, num_workers=0, drop_last=False)
                    iterator = iter(loader)
                    batch, previous = timed(lambda: next(iterator))
                    result["thread_loading_samples"].append({
                        "sample": sample, "threads": torch.get_num_threads(),
                        "fetch_seconds": previous, "image_ids": list(batch[2]),
                    })
                else:
                    continue
                break
        else:
            result["thread_comparison_skipped"] = "Current intraop thread count is already 1"
        complete = (
            sum(b["phase"] == "training" for b in result["pipeline_batches"]) == 2
            and sum(b["phase"] == "validation" for b in result["pipeline_batches"]) == 2
            and len(result["resident_gpu_step_seconds"]) == 3
            and (original_threads == 1 or len(result["thread_loading_samples"]) == 4)
        )
        result["status"] = "completed" if complete else "partial_budget_limited"
    except Exception as error:
        result["status"] = "failed"
        result["failures"].append(f"{type(error).__name__}: {error}")
        raise
    finally:
        # Restore the only temporary tuning change, even if loading/compute fails.
        try:
            torch.set_num_threads(original_threads)
        finally:
            result["restored_cpu_threads"] = torch.get_num_threads()
            result["cpu_threads_restored"] = torch.get_num_threads() == original_threads
            model = optimizer = resident = None
            result["disposable_model_and_optimizer_discarded"] = True
            result["elapsed_seconds"] = time.perf_counter() - started
            result["budget_overrun_seconds"] = max(0.0, result["elapsed_seconds"] - 90)
            result["resident_gpu_mean_seconds"] = (
                statistics.mean(result["resident_gpu_step_seconds"])
                if result["resident_gpu_step_seconds"] else None)
            output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
            print("Saved diagnostic timings:", output)
    return result


gpu_pipeline_timing_check = _run_gpu_pipeline_timing_check()
print(json.dumps(gpu_pipeline_timing_check, indent=2))

## 9. Prepared E0 experiment (disabled by default)

Every full-epoch and experiment-output operation is inside `if RUN_FULL_TRAINING`. With the default `False`, no E0 directory, checkpoint, history or curves are created. The smoke result must have passed before a deliberately enabled run. The fixed E0 directory is created exclusively: an existing directory stops the run instead of overwriting or resuming it.

For E0, reset seeds and instantiate fresh model, optimiser and loaders. Maximum 15 epochs; strict lower validation loss selects the best checkpoint. Early stopping uses three consecutive epochs without improvement. ReduceLROnPlateau uses validation loss, factor 0.5 and patience 2. Log the learning rate used that epoch and the next learning rate separately.

In [ ]:
E0_RUN_COMPLETED=False
if RUN_FULL_TRAINING:
    if smoke["status"]!="passed" or smoke["failures"] or not all(smoke["checks"].values()):
        raise RuntimeError("A successful GPU smoke check is required before enabling E0")
    if (globals().get("training_cache_check", {}).get("status") != "passed"
            or TRAIN_IMAGE_DIR != TRAIN_CACHE_DIR.resolve()):
        raise RuntimeError("A verified working-copy cache is required before E0")
    run_directory=WORKING_ROOT/E0_CONFIG["run_name"]
    # Exclusive creation prevents accidental overwrite, including interrupted runs.
    run_directory.mkdir(exist_ok=False)
    seed_everything(E0_CONFIG["seed"])
    experiment_model=ReferenceCNN().to(device=device,dtype=torch.float32)
    experiment_optimizer=torch.optim.Adam(experiment_model.parameters(),lr=E0_CONFIG["learning_rate"])
    experiment_criterion=nn.CrossEntropyLoss()
    experiment_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
        experiment_optimizer,mode=E0_CONFIG["scheduler"]["mode"],factor=E0_CONFIG["scheduler"]["factor"],
        patience=E0_CONFIG["scheduler"]["patience"],threshold=E0_CONFIG["scheduler"]["threshold"],threshold_mode="abs")
    experiment_train_dataset=FacialFeatureDataset(SETUP_ROOT,"train",training_image_directory=TRAIN_CACHE_DIR)
    experiment_validation_dataset=FacialFeatureDataset(SETUP_ROOT,"validation",training_image_directory=TRAIN_CACHE_DIR)
    experiment_train_loader=DataLoader(experiment_train_dataset,batch_size=E0_CONFIG["batch_size"],shuffle=True,
        generator=torch.Generator().manual_seed(E0_CONFIG["seed"]),num_workers=E0_CONFIG["num_workers"],drop_last=E0_CONFIG["drop_last"])
    experiment_validation_loader=DataLoader(experiment_validation_dataset,batch_size=E0_CONFIG["batch_size"],shuffle=False,num_workers=E0_CONFIG["num_workers"],drop_last=E0_CONFIG["drop_last"])
    split_checksum=file_hash(SETUP_ROOT/"data/processed/split_manifest.csv")
    if split_checksum!=artifact_hashes["data/processed/split_manifest.csv"]:
        raise ValueError("Saved split changed")
    configuration={**E0_CONFIG,"architecture":"Original ReferenceCNN: 94280 trainable parameters; eight raw logits",
        "class_mapping":mapping,"split_sha256":split_checksum,"artifact_sha256":artifact_hashes,
        "runtime_versions":report["versions"],"gpu":report["gpu"],
        "training_image_directory":str(TRAIN_CACHE_DIR.resolve()),"cache_report":str(TRAIN_CACHE_REPORT),
        "preprocessing":fingerprint_spec["preprocessing"],
        "training_augmentation":["RandomHorizontalFlip(p=0.5)","ColorJitter(brightness=0.1,contrast=0.1) before resize/padding"],
        "reproducibility":"Seeded, not guaranteed bitwise identical across versions or GPUs"}
    with (run_directory/"configuration.json").open("x",encoding="utf-8") as stream:
        stream.write(json.dumps(configuration,indent=2)+"\n")
    history=[]
    best_loss=float("inf")
    best_epoch=None
    no_improvement=0
    torch.cuda.synchronize(device)
    experiment_started=time.perf_counter()
    for epoch in range(1,E0_CONFIG["max_epochs"]+1):
        torch.cuda.synchronize(device)
        epoch_started=time.perf_counter()
        learning_rate=float(experiment_optimizer.param_groups[0]["lr"])
        training_metrics=train_epoch(experiment_model,experiment_train_loader,experiment_optimizer,experiment_criterion,device)
        validation_metrics=evaluate_epoch(experiment_model,experiment_validation_loader,experiment_criterion,device)
        if validation_metrics["loss"]<best_loss:
            best_loss=validation_metrics["loss"]
            best_epoch=epoch
            no_improvement=0
            torch.save({"model_state_dict":experiment_model.state_dict(),"class_mapping":mapping,
                        "configuration":configuration,"split_checksum":split_checksum,
                        "best_epoch":best_epoch,"best_validation_loss":best_loss},run_directory/"best_checkpoint.pt")
        else:
            no_improvement+=1
        experiment_scheduler.step(validation_metrics["loss"])
        torch.cuda.synchronize(device)
        row={"epoch":epoch,"training_loss":training_metrics["loss"],"validation_loss":validation_metrics["loss"],
             "training_accuracy":training_metrics["accuracy"],"validation_accuracy":validation_metrics["accuracy"],
             "training_samples":training_metrics["samples"],"validation_samples":validation_metrics["samples"],
             "learning_rate":learning_rate,"next_learning_rate":float(experiment_optimizer.param_groups[0]["lr"]),
             "epoch_elapsed_seconds":time.perf_counter()-epoch_started,
             "elapsed_seconds":time.perf_counter()-experiment_started}
        history.append(row)
        pd.DataFrame(history).to_csv(run_directory/"history.csv",index=False)
        print(row,flush=True)
        if no_improvement>=E0_CONFIG["early_stopping_patience"]:
            print("Early stopping: three consecutive epochs without lower validation loss")
            break
    if not all(file_hash(SETUP_ROOT/name)==digest for name,digest in artifact_hashes.items()):
        raise ValueError("Verified input artifacts changed during E0")
    E0_RUN_COMPLETED=True
    print("E0 outputs:",run_directory,"Best epoch:",best_epoch,"Best validation loss:",best_loss)
else:
    print("RUN_FULL_TRAINING=False: full epochs, E0 checkpoints and history are skipped.")


## 10. E0 loss/accuracy curves (also guarded)

Only a deliberately enabled, completed E0 run writes curves. No smoke diagnostics are plotted as learning curves.

In [ ]:
if RUN_FULL_TRAINING:
    if not E0_RUN_COMPLETED:
        raise RuntimeError("Complete the enabled E0 run before saving experimental curves")
    import matplotlib.pyplot as plt
    history_frame=pd.DataFrame(history)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for name,label in [("training","Training"),("validation","Validation")]:
        axes[0].plot(history_frame["epoch"],history_frame[name+"_loss"],label=label)
        axes[1].plot(history_frame["epoch"],history_frame[name+"_accuracy"],label=label)
    axes[0].set(xlabel="Epoch",ylabel="Cross-entropy loss",title="E0 loss")
    axes[1].set(xlabel="Epoch",ylabel="Accuracy",title="E0 accuracy",ylim=(0,1))
    for axis in axes:
        axis.legend(frameon=False)
        axis.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(run_directory/"loss_accuracy_curves.png",dpi=300)
    plt.show()
    plt.close(fig)
else:
    print("Full training disabled: no E0 learning curves or training-output files written.")


In [ ]:
# 11. EVALUATION ONLY — reload completed E0; no setup, copying or training.
RUN_FULL_TRAINING = False
try:
    import hashlib
    import inspect
    import json
    import math
    import platform
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    from torch.utils.data import DataLoader
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
except ImportError as error:
    raise RuntimeError("Evaluation dependency unavailable; no packages were installed") from error


def _e0_validation_statistics(true, predicted, loss_sum, names):
    """Explicit eight-class order; zero precision/F1 if a class is never predicted."""
    labels = list(range(len(names)))
    precision, recall, f1, support = precision_recall_fscore_support(
        true, predicted, labels=labels, zero_division=0)
    matrix = confusion_matrix(true, predicted, labels=labels)
    per_class = pd.DataFrame({"class_index": labels, "class": names,
        "precision": precision, "recall": recall, "f1": f1, "support": support})
    scores = {"samples": len(true), "cross_entropy": loss_sum / len(true),
        "accuracy": float(accuracy_score(true, predicted)),
        "macro_f1": float(f1.mean()),
        "weighted_f1": float(np.average(f1, weights=support))}
    return scores, per_class, matrix


def _evaluate_saved_e0_validation():
    required = ["ReferenceCNN", "FacialFeatureDataset", "SETUP_ROOT", "TRAIN_CACHE_DIR",
                "training_cache_check", "report", "artifact_hashes"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Required verified session variables unavailable: " + ", ".join(missing)
                           + ". Evaluation stopped; setup/training will not be rerun.")
    if (report.get("status") != "passed" or report.get("failures")
            or not report.get("gpu_check_executed") or not all(report["checks"].values())):
        raise RuntimeError("Existing GPU setup verification is not passed")
    if (training_cache_check.get("status") != "passed"
            or training_cache_check.get("verified_files") != 63660
            or training_cache_check.get("failures") or training_cache_check.get("hash_mismatches")):
        raise RuntimeError("Existing complete image-cache verification is unavailable")
    if not torch.cuda.is_available():
        raise RuntimeError("Verified cuda:0 is unavailable; evaluation stopped")
    gpu = torch.device("cuda:0")
    run = Path("/kaggle/working/E0_reference_cnn_seed42_attempt2")
    destination = run / "evaluation"
    if destination.exists() or destination.is_symlink():
        raise FileExistsError(f"Evaluation outputs already exist; left untouched: {destination}")
    paths = {"checkpoint": run / "best_checkpoint.pt", "configuration": run / "configuration.json",
             "history": run / "history.csv", "cache_report": run.parent / "training_image_cache_check.json"}
    paths.update({name: Path(SETUP_ROOT) / name for name in artifact_hashes})
    for name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Required existing file unavailable ({name}): {path}")

    def sha256(path):
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    before = {name: sha256(path) for name, path in paths.items()}
    config = json.loads(paths["configuration"].read_text())
    history = pd.read_csv(paths["history"], float_precision="round_trip")
    checkpoint = torch.load(paths["checkpoint"], map_location="cpu", weights_only=True)
    saved_mapping = json.loads((Path(SETUP_ROOT) / "data/processed/class_mapping.json").read_text())
    split = pd.read_csv(Path(SETUP_ROOT) / "data/processed/split_manifest.csv",
                        dtype=str, keep_default_na=False, na_filter=False)
    if (config["run_name"] != run.name or config["seed"] != 42 or config["max_epochs"] != 15
            or checkpoint["configuration"] != config or checkpoint["best_epoch"] != 15):
        raise ValueError("Checkpoint/configuration does not describe the completed E0 epoch 15")
    if (history["epoch"].tolist() != list(range(1, 16))
            or int(history.loc[history["validation_loss"].idxmin(), "epoch"]) != 15):
        raise ValueError("History does not contain 15 epochs with best validation loss at epoch 15")
    epoch15 = history.loc[history["epoch"].eq(15)].iloc[0]
    if (int(epoch15["validation_samples"]) != 12733
            or not math.isclose(checkpoint["best_validation_loss"], float(epoch15["validation_loss"]),
                                rel_tol=0, abs_tol=1e-12)):
        raise ValueError("Checkpoint best loss/support disagrees with saved history")
    if checkpoint["class_mapping"] != saved_mapping or config["class_mapping"] != saved_mapping:
        raise ValueError("Saved class mappings disagree")
    names = [saved_mapping["index_to_class"][str(i)] for i in range(8)]
    if names != sorted(names) or saved_mapping["class_to_index"] != dict(zip(names, range(8))):
        raise ValueError("Expected sorted, exact eight-class encoding")
    split_hash = before["data/processed/split_manifest.csv"]
    if (split_hash != config["split_sha256"] or split_hash != checkpoint["split_checksum"]
            or config["artifact_sha256"] != artifact_hashes
            or any(before[name] != digest for name, digest in artifact_hashes.items())):
        raise ValueError("Verified code/metadata or split checksum changed")
    for implementation, name in [(ReferenceCNN, "src/models.py"),
                                  (FacialFeatureDataset, "src/data_pipeline.py")]:
        if Path(inspect.getfile(implementation)).resolve() != paths[name].resolve():
            raise ValueError("Session code was not loaded from the verified setup")
    cache = Path(TRAIN_CACHE_DIR).resolve()
    cache_report = json.loads(paths["cache_report"].read_text())
    if (cache != (run.parent / "enge707_train_cache").resolve() or not cache.is_dir()
            or cache_report["status"] != "passed" or cache_report["verified_files"] != 63660
            or Path(cache_report["cache_directory"]).resolve() != cache):
        raise ValueError("Verified working-image cache is unavailable or changed")
    if (len(split) != 63660 or not split["image_id"].is_unique
            or not split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            or split["split"].value_counts().to_dict() != {"train": 50927, "validation": 12733}):
        raise ValueError("Saved split coverage/IDs changed")
    expected = split.loc[split["split"].eq("validation")].reset_index(drop=True)
    dataset = FacialFeatureDataset(SETUP_ROOT, "validation", training_image_directory=cache)
    if (len(dataset) != 12733 or dataset.augmentation is not None
            or not dataset.rows.equals(expected[["image_id", "class", "image_path"]])):
        raise ValueError("Validation dataset must retain saved membership/order without augmentation")
    expected_ids = expected["image_id"].tolist()
    expected_targets = expected["class"].map(saved_mapping["class_to_index"]).tolist()
    expected_support = [int(expected["class"].eq(name).sum()) for name in names]
    if not all(expected_support):
        raise ValueError("All eight classes must be present")
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0, drop_last=False)
    model = ReferenceCNN().to(device=gpu, dtype=torch.float32)
    if sum(p.numel() for p in model.parameters() if p.requires_grad) != 94280:
        raise ValueError("Reference architecture parameter count changed")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()
    # Reserve exclusively; an interrupted evaluation is never silently overwritten.
    destination.mkdir(exist_ok=False)
    ids, true, predicted, confidence = [], [], [], []
    loss_sum = 0.0
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        for batch_number, (images, labels, image_ids) in enumerate(loader, 1):
            start, end = len(ids), len(ids) + len(image_ids)
            if (list(image_ids) != expected_ids[start:end]
                    or labels.tolist() != expected_targets[start:end]):
                raise ValueError("Validation IDs/targets differ from saved order")
            if images.dtype != torch.float32 or labels.dtype != torch.int64:
                raise ValueError("Unexpected input/label precision")
            images, labels = images.to(gpu), labels.to(gpu)
            logits = model(images)
            loss = criterion(logits, labels)
            if logits.shape != (len(image_ids), 8) or not bool(torch.isfinite(logits).all() & torch.isfinite(loss)):
                raise FloatingPointError("Invalid logits/loss")
            maximum, indices = logits.softmax(dim=1).max(dim=1)
            loss_sum += float(loss) * len(image_ids)  # Include the final partial batch.
            ids.extend(image_ids)
            true.extend(labels.cpu().tolist())
            predicted.extend(indices.cpu().tolist())
            confidence.extend(maximum.cpu().tolist())
            if batch_number % 100 == 0 or batch_number == len(loader):
                print(f"Evaluation only: {len(ids):,}/12,733 validation images", flush=True)
    scores, per_class, matrix = _e0_validation_statistics(true, predicted, loss_sum, names)
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    checks = {
        "exact_id_coverage_and_order": ids == expected_ids and len(set(ids)) == 12733,
        "true_labels_match": true == expected_targets,
        "class_supports_match": per_class["support"].tolist() == expected_support,
        "prediction_indices_in_range": all(0 <= value < 8 for value in predicted),
        "finite_metrics_and_confidence": bool(np.isfinite(list(scores.values())).all()
            and np.isfinite(per_class[["precision", "recall", "f1"]].to_numpy()).all()
            and np.isfinite(confidence).all()),
        "confidence_in_range": all(0 <= value <= 1 for value in confidence),
        "confusion_total_matches": int(matrix.sum()) == 12733,
        "normalised_rows_sum_to_one": bool(np.allclose(normalised.sum(axis=1), 1)),
        "original_inputs_unchanged": all(sha256(path) == before[name] for name, path in paths.items()),
    }
    if not all(checks.values()):
        raise ValueError("Evaluation integrity checks failed: " + str(checks))
    comparison = {"epoch": 15, "history_accuracy": float(epoch15["validation_accuracy"]),
        "history_loss": float(epoch15["validation_loss"]),
        "accuracy_difference": scores["accuracy"] - float(epoch15["validation_accuracy"]),
        "loss_difference": scores["cross_entropy"] - float(epoch15["validation_loss"]),
        "accuracy_matches_exactly": scores["accuracy"] == float(epoch15["validation_accuracy"]),
        "loss_absolute_tolerance": 1e-6,
        "loss_matches_within_tolerance": math.isclose(scores["cross_entropy"],
            float(epoch15["validation_loss"]), rel_tol=0, abs_tol=1e-6)}
    discrepancy = not (comparison["accuracy_matches_exactly"] and comparison["loss_matches_within_tolerance"])
    predictions = pd.DataFrame({"image_id": ids, "true_class": [names[i] for i in true],
        "predicted_class": [names[i] for i in predicted], "max_softmax_confidence": confidence})
    with (destination / "validation_predictions.csv").open("x", newline="", encoding="utf-8") as stream:
        predictions.to_csv(stream, index=False)
    with (destination / "classification_report.csv").open("x", newline="", encoding="utf-8") as stream:
        per_class.to_csv(stream, index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), layout="constrained")
    for ax, values, title, fractional in zip(axes, [matrix, normalised],
            ["Raw counts", "Row-normalised (true class)"], [False, True]):
        image = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if fractional else matrix.max())
        ax.set(xticks=range(8), yticks=range(8), xticklabels=names, yticklabels=names,
               xlabel="Predicted class", ylabel="True class", title=title)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        for row in range(8):
            for column in range(8):
                value = values[row, column]
                ax.text(column, row, f"{value:.2f}" if fractional else str(value),
                        ha="center", va="center", fontsize=8.5,
                        color="white" if value > (0.5 if fractional else matrix.max() / 2) else "#25323A")
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.suptitle("E0 validation — saved checkpoint, epoch 15")
    with (destination / "confusion_matrix.png").open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)
    metrics = {"status": "passed_with_history_discrepancy" if discrepancy else "passed",
        "evaluation_only": True, "run_name": run.name, "best_epoch": 15,
        "device": str(gpu), "gpu": torch.cuda.get_device_name(0), "dtype": "float32",
        "batch_size": 32, "num_workers": 0, "drop_last": False, "shuffle": False,
        "augmentation": False, "trainable_parameters": 94280, **scores,
        "class_mapping": saved_mapping, "per_class": per_class.to_dict(orient="records"),
        "confusion_matrix": matrix.tolist(), "row_normalised_confusion_matrix": normalised.tolist(),
        "input_sha256": before, "checks": checks, "history_comparison": comparison,
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__),
                    "numpy": np.__version__, "pandas": pd.__version__},
        "warnings": ["Reloaded loss/accuracy differs from epoch 15; results were not altered."] if discrepancy else [],
        "limitations": ["No cross-version/GPU bitwise reproducibility claim.",
                        "Undefined per-class precision/F1 uses zero_division=0."]}
    with (destination / "validation_metrics.json").open("x", encoding="utf-8") as stream:
        json.dump(metrics, stream, indent=2, allow_nan=False)
        stream.write("\n")
    print(json.dumps({**scores, "history_comparison": comparison, "warnings": metrics["warnings"]}, indent=2))
    print("Saved evaluation only:", destination)
    return metrics


e0_validation_metrics = _evaluate_saved_e0_validation()